# HydrAI: ML Training Data Generation

This notebook generates training datasets for ML surrogate models by running multiple PFR simulations with varied parameters.

## Overview

The data generation process:
1. Parameter Sweep: Varies 6 key parameters (temperature, pressure, geometry, flow, heat flux)
2. Multiple Reactants: Supports ethane, propane, naphtha, n-hexane
Efficient Collection: Disables plots by default; optional duplicate export as CSV (`save_complete_csv`)
4. Periodic Saves: Saves data incrementally to prevent loss
5. Rich Features: Collects 245+ features per simulation point

## Features
- Configurable parameter ranges
- Latin Hypercube Sampling (LHS) for efficient parameter space coverage
- Random or full-grid sampling alternatives
- Progress tracking and time estimates
- Automatic data validation
- Metadata export for reproducibility

In [ ]:
# Setup and Imports
import sys
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from datetime import datetime

# Project root: notebooks live under notebooks/; from there, project root is one level up
current_dir = Path(os.getcwd())
if (current_dir / 'src').exists():
    project_root = current_dir
elif (current_dir.parent / 'src').exists():
    project_root = current_dir.parent
else:
    project_root = current_dir
os.chdir(project_root)  # so configs/, data/training/, mechanisms/ resolve correctly

# IMPORTANT: Import cantera BEFORE adding src to sys.path
# This prevents namespace conflict: without this, Python would find src/cantera/
# (our package) instead of the actual cantera library
import cantera as ct
print(f"Cantera version: {ct.__version__}")

# Suppress all Cantera/SUNDIALS warnings and messages
import warnings
import logging

# Add project root to path (after cantera is imported)
sys.path.insert(0, str(project_root))

from src.ml.data_generation import TrainingDataGenerator

print("HydrAI: ML Training Data Generation")
print("=" * 60)
print(f"Project root: {project_root}")

# Run control flags (metadata/data only)
IF_SAVE_METADATA = True  # Save metadata JSON when generating dataset
IF_SAVE_TRAINING_DATA = True  # Save training data (partial + final .pkl) to output_dir
IF_SAVE_COMPLETE_CSV = True   # Also write training_data_complete_*.csv (large); optional


## Step 1: Configuration

Configure the data generation parameters. You can either:
- Load from a JSON config file
- Set parameters directly in the notebook

Sampling method: `latin` (LHS), `random`, or `full_grid` / `structured_grid` / `grid`. Use `parameter_ranges` for grid; use `random_sample_bounds` for random/LHS.

In [ ]:
# Load configuration from JSON (edit configs/ml/ml_data_generation_config.json)
CONFIG_FILE = 'configs/ml/ml_data_generation_config.json'

if not Path(CONFIG_FILE).exists():
    raise FileNotFoundError(f"Config not found: {CONFIG_FILE}")

with open(CONFIG_FILE, 'r') as f:
    config = json.load(f)
print(f"[OK] Loaded configuration from: {CONFIG_FILE}")

REACTANTS = config.get('reactants', ['ethane'])
MAX_COMBINATIONS = config.get('max_combinations_per_reactant', 100)
OUTPUT_DIR = config.get('output_dir', 'data/training')
SAVE_INTERVAL = config.get('save_interval', 10)
PARAM_RANGES = config.get('parameter_ranges', {})
SAMPLING_METHOD = config.get('sampling_method', 'latin_hypercube')
LHS_SEED = config.get('lhs_seed', 42)
RANDOM_SAMPLE_BOUNDS = config.get('random_sample_bounds', None)

print(f"  - Reactants: {REACTANTS}")
print(f"  - Max combinations per reactant: {MAX_COMBINATIONS}")
print(f"  - Output directory: {OUTPUT_DIR}")
print(f"  - Sampling method: {SAMPLING_METHOD}")

print(f"\nParameter Ranges:")
for param, values in PARAM_RANGES.items():
    if isinstance(values, list) and len(values) == 3:
        print(f"  - {param}: {values[0]} to {values[1]} ({values[2]} points)")
    else:
        print(f"  - {param}: {values}")

## Step 2: Initialize Data Generator

Create the data generator with your configuration.

In [ ]:
# Initialize generator
generator = TrainingDataGenerator(output_dir=OUTPUT_DIR, disable_plots=True)

# Update parameter ranges from config
if PARAM_RANGES:
    for key, value in PARAM_RANGES.items():
        if isinstance(value, list) and len(value) == 3:
            # Convert [min, max, n_points] to numpy array
            generator.param_ranges[key] = np.linspace(value[0], value[1], value[2])
            print(f"[OK] Updated {key}: {len(generator.param_ranges[key])} points")

# Calculate total combinations
total_combinations = generator._calculate_total_combinations()
print(f"\n[OK] Data generator initialized")
print(f"  - Output directory: {generator.output_dir}")
print(f"  - Total possible combinations: {total_combinations:,}")
print(f"  - Will generate: {len(REACTANTS) * MAX_COMBINATIONS:,} simulations")

### Step 2.1: Sampling preview moved to Main_3

EDA and training-space coverage plotting are handled in `Main_3_data_exploration_feature_engineering.ipynb`.

Preview how the parameter space will be explored: 1D marginals (uniformity) and 2D pairs (space filling).

In [ ]:
print("[INFO] Sampling preview removed from Main_2 for cleaner generation workflow.")
print("       Use Main_3_data_exploration_feature_engineering.ipynb for all EDA/coverage plots.")

## Step 3: Generate Training Data

Run the data generation process. This will:
- Run multiple PFR simulations with varied parameters
- Collect features and targets at each simulation point
- Save data incrementally to prevent loss
- Generate metadata for reproducibility

Note: This may take 5-30 minutes depending on the number of simulations.

In [ ]:
# Generate dataset
print("=" * 60)
print("Starting Training Data Generation...")
print("=" * 60)
print(f"Reactants: {REACTANTS}")
print(f"Max combinations per reactant: {MAX_COMBINATIONS}")
print(f"Sampling method: {SAMPLING_METHOD}")
print(f"Save interval: {SAVE_INTERVAL} simulations")
print("=" * 60)

start_time = time.time()

dataset = generator.generate_dataset(
    reactants=REACTANTS,
    max_combinations_per_reactant=MAX_COMBINATIONS,
    save_interval=SAVE_INTERVAL,
    random_sample_bounds=RANDOM_SAMPLE_BOUNDS,
    sampling_method=SAMPLING_METHOD,
    lhs_seed=LHS_SEED,
    save_metadata=IF_SAVE_METADATA,
    save_training_data=IF_SAVE_TRAINING_DATA,
    save_complete_csv=IF_SAVE_COMPLETE_CSV,
)

elapsed_time = time.time() - start_time
hours = int(elapsed_time // 3600)
minutes = int((elapsed_time % 3600) // 60)
seconds = int(elapsed_time % 60)

print(f"\n[OK] Data generation completed!")
print(f"  - Total time: {hours:02d}:{minutes:02d}:{seconds:02d}")

# Calculate actual number of simulations completed
# The dataset has multiple rows per simulation (one per reactor step)
# We get the actual count from the metadata file
if dataset is not None and len(dataset) > 0:
    import glob
    metadata_files = sorted(glob.glob(str(generator.output_dir / 'metadata_*.json')), reverse=True)
    if metadata_files:
        with open(metadata_files[0], 'r') as f:
            metadata = json.load(f)
        n_simulations = metadata.get('total_simulations', len(REACTANTS) * MAX_COMBINATIONS)
    else:
        n_simulations = len(REACTANTS) * MAX_COMBINATIONS
    
    print(f"  - Simulations completed: {n_simulations}")
    print(f"  - Data points collected: {len(dataset):,}")
    if n_simulations > 0:
        print(f"  - Average time per simulation: {elapsed_time / n_simulations:.2f} seconds")
        print(f"  - Data points per simulation: {len(dataset) / n_simulations:.1f}")
    else:
        print("  - Average time: N/A (no simulations completed)")
else:
    print("  - Average time: N/A (no data collected)")
